In [ ]:
from azure.storage.blob import BlobServiceClient
from azure.core.credentials import AzureKeyCredential
from azure.ai.documentintelligence import DocumentIntelligenceClient
import base64
import json


# --- Preencha com seus dados ---
BLOB_CONN_STR = ""
CONTAINER = "pdfarquivos"
BLOB_PDF = "IMG_0104.jpg"
ENDPOINT = ""
KEY = ""
BLOB_SAIDA = "recibo.json"
# ------------------------------

def analyze_result_to_dict(result):
    # Extrai informação principal para salvar em JSON
    pages = []
    for page in result.pages:
        lines = [line.content for line in page.lines]
        pages.append({
            "page_number": page.page_number,
            "lines": lines
        })
    return {"pages": pages}

# Baixar o PDF do Blob Storage
blob_service = BlobServiceClient.from_connection_string(BLOB_CONN_STR)
blob_client = blob_service.get_container_client(CONTAINER).get_blob_client(BLOB_PDF)
pdf_data = blob_client.download_blob().readall()

# Converter para base64
pdf_base64 = base64.b64encode(pdf_data).decode("utf-8")

# Criar cliente Document Intelligence
client = DocumentIntelligenceClient(endpoint=ENDPOINT, credential=AzureKeyCredential(KEY))

# Iniciar análise do documento (base64Source)
poller = client.begin_analyze_document(
    "prebuilt-layout",  # ou "prebuilt-document"
    {"base64Source": pdf_base64}
)

# Obter resultado
result = poller.result()

# Converter resultado em dict manualmente
result_json = analyze_result_to_dict(result)

# Salvar JSON no Blob Storage
output_blob = blob_service.get_container_client(CONTAINER).get_blob_client(BLOB_SAIDA)
output_blob.upload_blob(json.dumps(result_json), overwrite=True)

# Exibir algumas linhas para validação rápida
print("Exemplo de linhas extraídas:")
for page in result.pages:
    print(f"Página {page.page_number}:")
    for line in page.lines[:3]:
        print(line.content)

Exemplo de linhas extraídas:
Página 1:
CHURRASCARIA LACADOR
CNPJ: 34.317.585/0001-90 CHURRASCARIA LACADOR LTDA
ROD BR316 KM 08, SN CENTRO ANANINDEUA-PA 67030-007
